In [1]:
import pandas as pd
import numpy as np

from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

In [2]:
customers = pd.read_csv("customers.csv")
products = pd.read_csv("products.csv")
transactions = pd.read_csv("transactions.csv")
inventory = pd.read_csv("inventory.csv")
campaigns = pd.read_csv("campaigns.csv")

In [3]:
print("customers:", customers.shape)
print("products:", products.shape)
print("transactions:", transactions.shape)
print("inventory:", inventory.shape)
print("campaigns:", campaigns.shape)

customers: (3000, 5)
products: (250, 5)
transactions: (18000, 10)
inventory: (5000, 7)
campaigns: (1200, 8)


In [4]:
print("CUSTOMERS")
print(customers.columns.tolist())

print("\nPRODUCTS")
print(products.columns.tolist())

print("\nTRANSACTIONS")
print(transactions.columns.tolist())

print("\nINVENTORY")
print(inventory.columns.tolist())

print("\nCAMPAIGNS")
print(campaigns.columns.tolist())

CUSTOMERS
['customer_id', 'region', 'customer_segment', 'signup_date', 'preferred_channel']

PRODUCTS
['product_id', 'category', 'unit_cost', 'list_price', 'supplier_lead_days']

TRANSACTIONS
['transaction_id', 'customer_id', 'product_id', 'timestamp', 'quantity', 'unit_price', 'discount_pct', 'status', 'channel', 'payment_method']

INVENTORY
['inventory_record_id', 'product_id', 'date', 'stock_on_hand', 'reorder_threshold', 'stockout_flag', 'supplier_delay_days']

CAMPAIGNS
['campaign_id', 'date', 'channel', 'category', 'spend', 'impressions', 'clicks', 'conversions']


In [5]:
display(customers.head())
display(products.head())
display(transactions.head())
display(inventory.head())
display(campaigns.head())

,customer_id,region,customer_segment,signup_date,preferred_channel
0,CU00001,West,Returning,2025-04-26,App
1,CU00002,Central,Loyal,2023-06-28,App
2,CU00003,East,Loyal,2023-10-18,App
3,CU00004,East,Returning,2024-11-02,Store
4,CU00005,West,New,2025-03-30,Web


,product_id,category,unit_cost,list_price,supplier_lead_days
0,PR0001,Sports,24.30,105.77,12
1,PR0002,Home,108.31,261.50,19
2,PR0003,Sports,223.15,186.62,5
3,PR0004,Sports,88.98,43.10,9
4,PR0005,Home,64.40,312.23,3


,transaction_id,customer_id,product_id,timestamp,quantity,unit_price,discount_pct,status,channel,payment_method
0,TX000001,CU01588,PR0103,2025-08-22 12:30:00,5,307.70,0.20,Completed,App,Card
1,TX000002,CU00382,PR0101,2025-06-06 04:03:00,4,346.98,0.00,Completed,Web,Card
2,TX000003,CU01799,PR0176,2025-06-24 17:21:00,2,33.87,0.10,Returned,Web,Wallet
3,TX000004,CU01643,PR0116,2025-05-16 13:57:00,2,166.19,0.05,Pending,App,Wallet
4,TX000005,CU01792,PR0022,2025-06-18 02:13:00,5,361.56,0.05,Completed,Store,Cash


,inventory_record_id,product_id,date,stock_on_hand,reorder_threshold,stockout_flag,supplier_delay_days
0,IN000001,PR0160,2025-03-16,221,23,0,18
1,IN000002,PR0076,2025-03-07,142,65,0,9
2,IN000003,PR0113,2025-11-02,203,38,0,11
3,IN000004,PR0206,2025-07-13,195,12,0,15
4,IN000005,PR0244,2025-09-29,178,50,0,1


,campaign_id,date,channel,category,spend,impressions,clicks,conversions
0,CA0001,2025-04-26,Email,Electronics,3467.47,67776,5355,1047
1,CA0002,2025-06-28,Web,Home,1974.23,32286,5864,1084
2,CA0003,2025-01-22,App,Home,7577.51,69685,9452,949
3,CA0004,2025-10-30,Email,Home,138.93,5470,7840,1000
4,CA0005,2025-12-17,App,Beauty,2730.23,99159,1529,1075


In [6]:
for name, df in {
    "customers": customers,
    "products": products,
    "transactions": transactions,
    "inventory": inventory,
    "campaigns": campaigns,
}.items():
    print("\n", "=" * 50)
    print(name.upper())
    print("Shape:", df.shape)
    print("Duplicates:", df.duplicated().sum())
    print("Missing values:")
    print(df.isna().sum().sort_values(ascending=False).head(20))


CUSTOMERS
Shape: (3000, 5)
Duplicates: 0
Missing values:
customer_id          0
region               0
customer_segment     0
signup_date          0
preferred_channel    0
dtype: int64

PRODUCTS
Shape: (250, 5)
Duplicates: 0
Missing values:
product_id            0
category              0
unit_cost             0
list_price            0
supplier_lead_days    0
dtype: int64

TRANSACTIONS
Shape: (18000, 10)
Duplicates: 0
Missing values:
transaction_id    0
customer_id       0
product_id        0
timestamp         0
quantity          0
unit_price        0
discount_pct      0
status            0
channel           0
payment_method    0
dtype: int64

INVENTORY
Shape: (5000, 7)
Duplicates: 0
Missing values:
inventory_record_id    0
product_id             0
date                   0
stock_on_hand          0
reorder_threshold      0
stockout_flag          0
supplier_delay_days    0
dtype: int64

CAMPAIGNS
Shape: (1200, 8)
Duplicates: 0
Missing values:
campaign_id    0
date           0
channel    

In [7]:
transactions["status"].value_counts()

status
Completed    13962
Cancelled     1689
Returned      1431
Pending        918
Name: count, dtype: int64

In [8]:
transactions["status"].value_counts(normalize=True) * 100

status
Completed    77.566667
Cancelled     9.383333
Returned      7.950000
Pending       5.100000
Name: proportion, dtype: float64

In [9]:
products[
    [
        "product_id",
        "category",
        "unit_cost",
        "list_price",
        "supplier_lead_days",
    ]
].head(10)

,product_id,category,unit_cost,list_price,supplier_lead_days
0,PR0001,Sports,24.30,105.77,12
1,PR0002,Home,108.31,261.50,19
2,PR0003,Sports,223.15,186.62,5
3,PR0004,Sports,88.98,43.10,9
4,PR0005,Home,64.40,312.23,3
5,PR0006,Sports,125.69,176.34,10
6,PR0007,Fashion,135.66,224.50,4
7,PR0008,Beauty,81.30,35.94,17
8,PR0009,Electronics,92.24,317.50,2
9,PR0010,Beauty,99.91,381.92,4


In [10]:
products["price_below_cost"] = (
    products["list_price"] < products["unit_cost"]
)

print(
    "Products with list price below unit cost:",
    products["price_below_cost"].sum()
)

print(
    "Percentage:",
    round(products["price_below_cost"].mean() * 100, 2),
    "%"
)

Products with list price below unit cost: 76
Percentage: 30.4 %


In [11]:
products.loc[
    products["price_below_cost"],
    [
        "product_id",
        "category",
        "unit_cost",
        "list_price",
    ]
].head(20)

,product_id,category,unit_cost,list_price
2,PR0003,Sports,223.15,186.62
3,PR0004,Sports,88.98,43.10
7,PR0008,Beauty,81.30,35.94
11,PR0012,Beauty,117.83,12.60
13,PR0014,Grocery,197.73,80.41
16,PR0017,Grocery,200.79,180.90
19,PR0020,Fashion,215.65,78.29
20,PR0021,Beauty,236.46,64.48
26,PR0027,Electronics,79.36,66.51
27,PR0028,Grocery,195.92,11.42


In [12]:
tx = (
    transactions
    .merge(
        customers,
        on="customer_id",
        how="left"
    )
    .merge(
        products,
        on="product_id",
        how="left"
    )
)

In [13]:
print("Original transactions:", transactions.shape)
print("Joined table:", tx.shape)

Original transactions: (18000, 10)
Joined table: (18000, 19)


In [14]:
display(tx.head())

,transaction_id,customer_id,product_id,timestamp,quantity,unit_price,discount_pct,status,channel,payment_method,region,customer_segment,signup_date,preferred_channel,category,unit_cost,list_price,supplier_lead_days,price_below_cost
0,TX000001,CU01588,PR0103,2025-08-22 12:30:00,5,307.70,0.20,Completed,App,Card,West,New,2023-09-05,App,Electronics,63.55,384.62,8,False
1,TX000002,CU00382,PR0101,2025-06-06 04:03:00,4,346.98,0.00,Completed,Web,Card,North,Returning,2024-06-15,Web,Home,15.38,346.98,18,False
2,TX000003,CU01799,PR0176,2025-06-24 17:21:00,2,33.87,0.10,Returned,Web,Wallet,East,New,2023-08-31,Web,Grocery,162.77,37.63,21,True
3,TX000004,CU01643,PR0116,2025-05-16 13:57:00,2,166.19,0.05,Pending,App,Wallet,Central,New,2023-12-19,Web,Grocery,124.90,174.94,23,False
4,TX000005,CU01792,PR0022,2025-06-18 02:13:00,5,361.56,0.05,Completed,Store,Cash,East,Loyal,2023-04-11,Store,Home,104.19,380.59,7,False


In [15]:
assert len(tx) == len(transactions)

print("Row count preserved:", len(tx))

Row count preserved: 18000


In [16]:
print(
    "Missing joined customer regions:",
    tx["region"].isna().sum()
)

print(
    "Missing joined customer segments:",
    tx["customer_segment"].isna().sum()
)

Missing joined customer regions: 0
Missing joined customer segments: 0


In [17]:
print(
    "Missing joined product categories:",
    tx["category"].isna().sum()
)

print(
    "Missing joined unit costs:",
    tx["unit_cost"].isna().sum()
)

Missing joined product categories: 0
Missing joined unit costs: 0


In [18]:
tx.isna().sum().sort_values(ascending=False)

transaction_id        0
customer_id           0
product_id            0
timestamp             0
quantity              0
unit_price            0
discount_pct          0
status                0
channel               0
payment_method        0
region                0
customer_segment      0
signup_date           0
preferred_channel     0
category              0
unit_cost             0
list_price            0
supplier_lead_days    0
price_below_cost      0
dtype: int64

In [19]:
tx[
    [
        "transaction_id",
        "customer_id",
        "product_id",
        "region",
        "customer_segment",
        "category",
        "unit_cost",
        "unit_price",
        "discount_pct",
        "status",
    ]
].head(10)

,transaction_id,customer_id,product_id,region,customer_segment,category,unit_cost,unit_price,discount_pct,status
0,TX000001,CU01588,PR0103,West,New,Electronics,63.55,307.70,0.20,Completed
1,TX000002,CU00382,PR0101,North,Returning,Home,15.38,346.98,0.00,Completed
2,TX000003,CU01799,PR0176,East,New,Grocery,162.77,33.87,0.10,Returned
3,TX000004,CU01643,PR0116,Central,New,Grocery,124.90,166.19,0.05,Pending
4,TX000005,CU01792,PR0022,East,Loyal,Home,104.19,361.56,0.05,Completed
5,TX000006,CU02380,PR0148,West,At-risk,Electronics,47.79,147.69,0.15,Completed
6,TX000007,CU01351,PR0024,South,At-risk,Electronics,19.06,329.91,0.10,Completed
7,TX000008,CU02962,PR0115,West,At-risk,Home,214.37,245.77,0.10,Completed
8,TX000009,CU01772,PR0056,South,Returning,Beauty,66.65,221.56,0.15,Pending
9,TX000010,CU00994,PR0095,West,Returning,Electronics,143.39,18.66,0.00,Completed


In [20]:
tx["revenue"] = (
    tx["quantity"]
    * tx["unit_price"]
    * (1 - tx["discount_pct"])
)

In [21]:
tx[
    [
        "quantity",
        "unit_price",
        "discount_pct",
        "revenue"
    ]
].head(10)

,quantity,unit_price,discount_pct,revenue
0,5,307.70,0.20,1230.800
1,4,346.98,0.00,1387.920
2,2,33.87,0.10,60.966
3,2,166.19,0.05,315.761
4,5,361.56,0.05,1717.410
5,4,147.69,0.15,502.146
6,5,329.91,0.10,1484.595
7,2,245.77,0.10,442.386
8,1,221.56,0.15,188.326
9,2,18.66,0.00,37.320


In [22]:
print(
    "Total discounted transaction value:",
    f"₹{tx['revenue'].sum():,.2f}"
)

Total discounted transaction value: ₹9,390,803.10


In [23]:
tx["estimated_cost"] = (
    tx["quantity"]
    * tx["unit_cost"]
)

tx["estimated_margin"] = (
    tx["revenue"]
    - tx["estimated_cost"]
)

tx["negative_margin"] = (
    tx["estimated_margin"] < 0
).astype(int)

In [24]:
tx[
    [
        "transaction_id",
        "quantity",
        "unit_price",
        "discount_pct",
        "unit_cost",
        "revenue",
        "estimated_cost",
        "estimated_margin",
        "negative_margin"
    ]
].head(10)

,transaction_id,quantity,unit_price,discount_pct,unit_cost,revenue,estimated_cost,estimated_margin,negative_margin
0,TX000001,5,307.70,0.20,63.55,1230.800,317.75,913.050,0
1,TX000002,4,346.98,0.00,15.38,1387.920,61.52,1326.400,0
2,TX000003,2,33.87,0.10,162.77,60.966,325.54,-264.574,1
3,TX000004,2,166.19,0.05,124.90,315.761,249.80,65.961,0
4,TX000005,5,361.56,0.05,104.19,1717.410,520.95,1196.460,0
5,TX000006,4,147.69,0.15,47.79,502.146,191.16,310.986,0
6,TX000007,5,329.91,0.10,19.06,1484.595,95.30,1389.295,0
7,TX000008,2,245.77,0.10,214.37,442.386,428.74,13.646,0
8,TX000009,1,221.56,0.15,66.65,188.326,66.65,121.676,0
9,TX000010,2,18.66,0.00,143.39,37.320,286.78,-249.460,1


In [25]:
negative_margin_count = tx["negative_margin"].sum()

negative_margin_rate = (
    tx["negative_margin"].mean() * 100
)

print(
    "Transactions with apparent negative margin:",
    negative_margin_count
)

print(
    "Apparent negative-margin rate:",
    f"{negative_margin_rate:.2f}%"
)

Transactions with apparent negative margin: 6829
Apparent negative-margin rate: 37.94%


In [26]:
tx.loc[
    tx["negative_margin"] == 1,
    [
        "transaction_id",
        "product_id",
        "category",
        "quantity",
        "unit_price",
        "discount_pct",
        "unit_cost",
        "revenue",
        "estimated_cost",
        "estimated_margin"
    ]
].head(20)

,transaction_id,product_id,category,quantity,unit_price,discount_pct,unit_cost,revenue,estimated_cost,estimated_margin
2,TX000003,PR0176,Grocery,2,33.87,0.10,162.77,60.9660,325.54,-264.5740
9,TX000010,PR0095,Electronics,2,18.66,0.00,143.39,37.3200,286.78,-249.4600
11,TX000012,PR0072,Sports,5,35.47,0.00,39.69,177.3500,198.45,-21.1000
16,TX000017,PR0246,Fashion,1,10.76,0.20,27.79,8.6080,27.79,-19.1820
17,TX000018,PR0246,Fashion,4,11.43,0.15,27.79,38.8620,111.16,-72.2980
19,TX000020,PR0209,Grocery,2,142.93,0.00,181.77,285.8600,363.54,-77.6800
20,TX000021,PR0080,Home,3,47.20,0.15,231.12,120.3600,693.36,-573.0000
29,TX000030,PR0152,Beauty,5,164.81,0.10,195.87,741.6450,979.35,-237.7050
39,TX000040,PR0013,Home,4,195.23,0.05,191.21,741.8740,764.84,-22.9660
44,TX000045,PR0033,Electronics,5,116.94,0.05,193.88,555.4650,969.40,-413.9350


In [27]:
cancelled_value = tx.loc[
    tx["status"] == "Cancelled",
    "revenue"
].sum()

returned_value = tx.loc[
    tx["status"] == "Returned",
    "revenue"
].sum()

combined_exposure = (
    cancelled_value
    + returned_value
)

In [28]:
print(
    f"Cancelled transaction value: ₹{cancelled_value:,.2f}"
)

print(
    f"Returned transaction value: ₹{returned_value:,.2f}"
)

print(
    f"Combined cancelled + returned exposure: "
    f"₹{combined_exposure:,.2f}"
)

Cancelled transaction value: ₹861,849.61
Returned transaction value: ₹760,322.30
Combined cancelled + returned exposure: ₹1,622,171.91


In [29]:
non_pending_value = tx.loc[
    tx["status"] != "Pending",
    "revenue"
].sum()

exposure_pct = (
    combined_exposure
    / non_pending_value
    * 100
)

print(
    f"Cancelled + returned exposure as % of "
    f"non-pending transaction value: "
    f"{exposure_pct:.2f}%"
)

Cancelled + returned exposure as % of non-pending transaction value: 18.18%


In [30]:
tx["failed_transaction"] = (
    tx["status"].isin(["Cancelled", "Returned"])
).astype(int)

print(
    "Overall cancelled/returned rate:",
    f"{tx['failed_transaction'].mean() * 100:.2f}%"
)

Overall cancelled/returned rate: 17.33%


In [31]:
failure_by_category = (
    tx.groupby("category")
    .agg(
        transactions=("transaction_id", "count"),
        failure_rate=("failed_transaction", "mean"),
    )
    .reset_index()
)

failure_by_category["failure_rate"] *= 100

failure_by_category = failure_by_category.sort_values(
    "failure_rate",
    ascending=False
)

display(failure_by_category)

,category,transactions,failure_rate
4,Home,3141,17.765043
3,Grocery,3150,17.523810
1,Electronics,3286,17.468046
2,Fashion,2598,17.244034
5,Sports,3041,17.066754
0,Beauty,2784,16.846264


In [32]:
failed_value_by_category = (
    tx.loc[tx["failed_transaction"] == 1]
    .groupby("category")["revenue"]
    .sum()
    .reset_index(name="failed_value")
    .sort_values("failed_value", ascending=False)
)

display(failed_value_by_category)

,category,failed_value
4,Home,329781.5670
3,Grocery,287418.0280
1,Electronics,285988.0925
5,Sports,283834.1090
2,Fashion,234269.3495
0,Beauty,200880.7640


In [33]:
margin_by_category = (
    tx.groupby("category")
    .agg(
        transactions=("transaction_id", "count"),
        negative_margin_rate=("negative_margin", "mean"),
        avg_margin=("estimated_margin", "mean"),
    )
    .reset_index()
)

margin_by_category["negative_margin_rate"] *= 100

margin_by_category = margin_by_category.sort_values(
    "negative_margin_rate",
    ascending=False
)

display(margin_by_category)

,category,transactions,negative_margin_rate,avg_margin
0,Beauty,2784,55.208333,24.804606
3,Grocery,3150,40.507937,127.803119
1,Electronics,3286,37.279367,159.956494
2,Fashion,2598,33.949192,184.956561
4,Home,3141,32.951289,179.191841
5,Sports,3041,28.740546,163.899710


In [34]:
margin_by_discount = (
    tx.groupby("discount_pct")
    .agg(
        transactions=("transaction_id", "count"),
        negative_margin_rate=("negative_margin", "mean"),
        avg_margin=("estimated_margin", "mean"),
    )
    .reset_index()
)

margin_by_discount["negative_margin_rate"] *= 100

margin_by_discount = margin_by_discount.sort_values(
    "discount_pct"
)

display(margin_by_discount)

,discount_pct,transactions,negative_margin_rate,avg_margin
0,0.00,6250,30.208000,238.416859
1,0.05,2667,33.783277,180.232318
2,0.10,3623,38.862821,116.880111
3,0.15,2733,43.980973,61.623306
4,0.20,1815,51.680441,5.804365
5,0.25,912,53.947368,-37.458766


In [35]:
campaigns["invalid_clicks"] = (
    campaigns["clicks"] > campaigns["impressions"]
)

campaigns["invalid_conversions"] = (
    campaigns["conversions"] > campaigns["clicks"]
)

print(
    "Clicks > impressions:",
    campaigns["invalid_clicks"].sum()
)

print(
    "Conversions > clicks:",
    campaigns["invalid_conversions"].sum()
)

Clicks > impressions: 84
Conversions > clicks: 88


In [36]:
campaigns["campaign_invalid"] = (
    campaigns["invalid_clicks"]
    | campaigns["invalid_conversions"]
)

print(
    "Campaign rows with at least one issue:",
    campaigns["campaign_invalid"].sum()
)

Campaign rows with at least one issue: 171


In [37]:
inventory["stockout_above_threshold"] = (
    (inventory["stockout_flag"] == 1)
    & (
        inventory["stock_on_hand"]
        > inventory["reorder_threshold"]
    )
)

inventory["zero_stock_not_flagged"] = (
    (inventory["stock_on_hand"] == 0)
    & (inventory["stockout_flag"] == 0)
)

print(
    "Stockout flagged while stock > reorder threshold:",
    inventory["stockout_above_threshold"].sum()
)

print(
    "Zero stock but not marked stockout:",
    inventory["zero_stock_not_flagged"].sum()
)

Stockout flagged while stock > reorder threshold: 648
Zero stock but not marked stockout: 10


In [38]:
print(
    "Total stockout flags:",
    inventory["stockout_flag"].sum()
)

Total stockout flags: 760


In [39]:
product_tx = (
    tx.groupby("product_id")
    .agg(
        transaction_count=("transaction_id", "count"),

        avg_order_value=("revenue", "mean"),

        total_transaction_value=("revenue", "sum"),

        avg_discount=("discount_pct", "mean"),

        cancel_rate=(
            "status",
            lambda x: (x == "Cancelled").mean()
        ),

        return_rate=(
            "status",
            lambda x: (x == "Returned").mean()
        ),

        failure_rate=(
            "failed_transaction",
            "mean"
        ),

        negative_margin_rate=(
            "negative_margin",
            "mean"
        ),

        avg_margin=(
            "estimated_margin",
            "mean"
        ),
    )
    .reset_index()
)

display(product_tx.head())

,product_id,transaction_count,avg_order_value,total_transaction_value,avg_discount,cancel_rate,return_rate,failure_rate,negative_margin_rate,avg_margin
0,PR0001,69,251.834326,17376.5685,0.076812,0.086957,0.072464,0.159420,0.0,185.273457
1,PR0002,71,668.150972,47438.7190,0.084507,0.154930,0.014085,0.169014,0.0,341.695479
2,PR0003,67,484.122254,32436.1910,0.081343,0.074627,0.029851,0.104478,1.0,-205.311328
3,PR0004,65,125.277723,8143.0520,0.071538,0.107692,0.123077,0.230769,1.0,-169.040738
4,PR0005,60,780.708542,46842.5125,0.075833,0.100000,0.083333,0.183333,0.0,593.948542


In [40]:
print(product_tx.shape)

(250, 10)


In [41]:
product_inventory = (
    inventory.groupby("product_id")
    .agg(
        avg_stock_on_hand=(
            "stock_on_hand",
            "mean"
        ),

        avg_reorder_threshold=(
            "reorder_threshold",
            "mean"
        ),

        stockout_rate=(
            "stockout_flag",
            "mean"
        ),

        avg_supplier_delay=(
            "supplier_delay_days",
            "mean"
        ),

        stockout_inconsistency_rate=(
            "stockout_above_threshold",
            "mean"
        ),
    )
    .reset_index()
)

display(product_inventory.head())

,product_id,avg_stock_on_hand,avg_reorder_threshold,stockout_rate,avg_supplier_delay,stockout_inconsistency_rate
0,PR0001,170.500000,43.538462,0.192308,10.384615,0.192308
1,PR0002,164.900000,42.650000,0.200000,9.700000,0.200000
2,PR0003,113.565217,40.913043,0.217391,9.304348,0.217391
3,PR0004,143.814815,51.518519,0.074074,9.629630,0.037037
4,PR0005,177.250000,43.625000,0.312500,12.437500,0.312500


In [42]:
print(product_inventory.shape)

(250, 6)


In [43]:
product_risk = (
    products
    .merge(
        product_tx,
        on="product_id",
        how="left"
    )
    .merge(
        product_inventory,
        on="product_id",
        how="left"
    )
)

In [44]:
print(product_risk.shape)
display(product_risk.head())

(250, 20)


,product_id,category,unit_cost,list_price,supplier_lead_days,price_below_cost,transaction_count,avg_order_value,total_transaction_value,avg_discount,cancel_rate,return_rate,failure_rate,negative_margin_rate,avg_margin,avg_stock_on_hand,avg_reorder_threshold,stockout_rate,avg_supplier_delay,stockout_inconsistency_rate
0,PR0001,Sports,24.30,105.77,12,False,69,251.834326,17376.5685,0.076812,0.086957,0.072464,0.159420,0.0,185.273457,170.500000,43.538462,0.192308,10.384615,0.192308
1,PR0002,Home,108.31,261.50,19,False,71,668.150972,47438.7190,0.084507,0.154930,0.014085,0.169014,0.0,341.695479,164.900000,42.650000,0.200000,9.700000,0.200000
2,PR0003,Sports,223.15,186.62,5,True,67,484.122254,32436.1910,0.081343,0.074627,0.029851,0.104478,1.0,-205.311328,113.565217,40.913043,0.217391,9.304348,0.217391
3,PR0004,Sports,88.98,43.10,9,True,65,125.277723,8143.0520,0.071538,0.107692,0.123077,0.230769,1.0,-169.040738,143.814815,51.518519,0.074074,9.629630,0.037037
4,PR0005,Home,64.40,312.23,3,False,60,780.708542,46842.5125,0.075833,0.100000,0.083333,0.183333,0.0,593.948542,177.250000,43.625000,0.312500,12.437500,0.312500


In [45]:
product_risk["base_price_margin"] = (
    product_risk["list_price"]
    - product_risk["unit_cost"]
)

In [46]:
product_risk["base_margin_pct"] = (
    product_risk["base_price_margin"]
    / product_risk["unit_cost"]
)

In [47]:
product_risk[
    [
        "product_id",
        "category",
        "unit_cost",
        "list_price",
        "base_price_margin",
        "base_margin_pct"
    ]
].head()

,product_id,category,unit_cost,list_price,base_price_margin,base_margin_pct
0,PR0001,Sports,24.30,105.77,81.47,3.352675
1,PR0002,Home,108.31,261.50,153.19,1.414366
2,PR0003,Sports,223.15,186.62,-36.53,-0.163702
3,PR0004,Sports,88.98,43.10,-45.88,-0.515621
4,PR0005,Home,64.40,312.23,247.83,3.848292


In [48]:
ml_features = [
    "transaction_count",
    "avg_order_value",
    "avg_discount",
    "cancel_rate",
    "return_rate",
    "failure_rate",
    "negative_margin_rate",
    "avg_margin",
    "avg_stock_on_hand",
    "avg_reorder_threshold",
    "stockout_rate",
    "avg_supplier_delay",
    "stockout_inconsistency_rate",
    "supplier_lead_days",
    "base_margin_pct",
]

In [49]:
X = product_risk[ml_features].copy()

display(X.head())

,transaction_count,avg_order_value,avg_discount,cancel_rate,return_rate,failure_rate,negative_margin_rate,avg_margin,avg_stock_on_hand,avg_reorder_threshold,stockout_rate,avg_supplier_delay,stockout_inconsistency_rate,supplier_lead_days,base_margin_pct
0,69,251.834326,0.076812,0.086957,0.072464,0.159420,0.0,185.273457,170.500000,43.538462,0.192308,10.384615,0.192308,12,3.352675
1,71,668.150972,0.084507,0.154930,0.014085,0.169014,0.0,341.695479,164.900000,42.650000,0.200000,9.700000,0.200000,19,1.414366
2,67,484.122254,0.081343,0.074627,0.029851,0.104478,1.0,-205.311328,113.565217,40.913043,0.217391,9.304348,0.217391,5,-0.163702
3,65,125.277723,0.071538,0.107692,0.123077,0.230769,1.0,-169.040738,143.814815,51.518519,0.074074,9.629630,0.037037,9,-0.515621
4,60,780.708542,0.075833,0.100000,0.083333,0.183333,0.0,593.948542,177.250000,43.625000,0.312500,12.437500,0.312500,3,3.848292


In [50]:
X.isna().sum()

transaction_count              0
avg_order_value                0
avg_discount                   0
cancel_rate                    0
return_rate                    0
failure_rate                   0
negative_margin_rate           0
avg_margin                     0
avg_stock_on_hand              0
avg_reorder_threshold          0
stockout_rate                  0
avg_supplier_delay             0
stockout_inconsistency_rate    0
supplier_lead_days             0
base_margin_pct                0
dtype: int64

In [51]:
X = X.replace(
    [np.inf, -np.inf],
    np.nan
)

X = X.fillna(
    X.median()
)

In [52]:
print(
    "Missing ML values:",
    X.isna().sum().sum()
)

Missing ML values: 0


In [53]:
print(X.shape)

(250, 15)


In [54]:
scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)

print(X_scaled.shape)

(250, 15)


In [55]:
model = IsolationForest(
    n_estimators=300,
    contamination=0.10,
    random_state=42
)

product_risk["anomaly_label"] = model.fit_predict(X_scaled)

product_risk["raw_anomaly_score"] = model.decision_function(X_scaled)

In [56]:
product_risk["anomaly_label"].value_counts()

anomaly_label
 1    225
-1     25
Name: count, dtype: int64

In [57]:
product_risk["anomaly_score"] = (
    -product_risk["raw_anomaly_score"]
)

In [58]:
score_min = product_risk["anomaly_score"].min()
score_max = product_risk["anomaly_score"].max()

product_risk["risk_score"] = (
    (
        product_risk["anomaly_score"]
        - score_min
    )
    /
    (
        score_max
        - score_min
    )
    * 100
)

In [59]:
top_risky_products = (
    product_risk
    .sort_values(
        "risk_score",
        ascending=False
    )
)

display(
    top_risky_products[
        [
            "product_id",
            "category",
            "risk_score",
            "transaction_count",
            "failure_rate",
            "negative_margin_rate",
            "avg_discount",
            "avg_margin",
            "stockout_rate",
            "avg_supplier_delay",
            "base_margin_pct"
        ]
    ].head(20)
)

,product_id,category,risk_score,transaction_count,failure_rate,negative_margin_rate,avg_discount,avg_margin,stockout_rate,avg_supplier_delay,base_margin_pct
77,PR0078,Fashion,100.000000,67,0.164179,0.000000,0.073134,1011.307127,0.157895,11.105263,89.040767
188,PR0189,Sports,92.079639,73,0.287671,1.000000,0.078767,-511.125308,0.037037,9.185185,-0.951461
105,PR0106,Home,81.876387,68,0.250000,0.000000,0.082353,760.690397,0.000000,12.400000,6.427705
145,PR0146,Electronics,78.290296,91,0.241758,0.000000,0.075824,785.218176,0.285714,10.714286,9.535337
231,PR0232,Electronics,76.953298,70,0.228571,0.557143,0.070714,-26.086436,0.062500,12.500000,0.100347
23,PR0024,Electronics,76.872528,69,0.115942,0.000000,0.092029,873.963674,0.052632,11.631579,18.232424
39,PR0040,Fashion,76.129477,53,0.245283,0.000000,0.078302,375.870698,0.166667,9.944444,32.371917
8,PR0009,Electronics,75.417797,70,0.100000,0.000000,0.085714,540.515814,0.380952,10.666667,2.442108
79,PR0080,Home,74.330827,71,0.183099,1.000000,0.069014,-540.811641,0.333333,11.303030,-0.759735
100,PR0101,Home,71.837877,83,0.120482,0.000000,0.075904,858.820169,0.076923,6.153846,21.560468


In [60]:
product_risk["risk_level"] = pd.cut(
    product_risk["risk_score"],
    bins=[-1, 40, 70, 101],
    labels=[
        "Low",
        "Medium",
        "High"
    ]
)

In [61]:
product_risk["risk_level"].value_counts()

risk_level
Low       153
Medium     84
High       13
Name: count, dtype: int64

In [63]:
def explain_product(row):
    reasons = []

    if row["negative_margin_rate"] > product_risk["negative_margin_rate"].quantile(0.75):
        reasons.append("high apparent negative-margin rate")

    if row["failure_rate"] > product_risk["failure_rate"].quantile(0.75):
        reasons.append("high cancelled/returned rate")

    if row["avg_discount"] > product_risk["avg_discount"].quantile(0.75):
        reasons.append("high average discount")

    if row["stockout_rate"] > product_risk["stockout_rate"].quantile(0.75):
        reasons.append("high stockout flag rate")

    if row["avg_supplier_delay"] > product_risk["avg_supplier_delay"].quantile(0.75):
        reasons.append("high supplier delay")

    if row["base_margin_pct"] < product_risk["base_margin_pct"].quantile(0.25):
        reasons.append("weak supplied price-cost margin")

    if not reasons:
        reasons.append("unusual combination of business metrics")

    return "; ".join(reasons)

In [64]:
product_risk["why_flagged"] = product_risk.apply(
    explain_product,
    axis=1
)

In [65]:
display(
    product_risk
    .sort_values(
        "risk_score",
        ascending=False
    )
    [
        [
            "product_id",
            "category",
            "risk_score",
            "risk_level",
            "why_flagged"
        ]
    ]
    .head(20)
)

,product_id,category,risk_score,risk_level,why_flagged
77,PR0078,Fashion,100.000000,High,high supplier delay
188,PR0189,Sports,92.079639,High,high cancelled/returned rate; weak supplied pr...
105,PR0106,Home,81.876387,High,high cancelled/returned rate; high supplier delay
145,PR0146,Electronics,78.290296,High,high cancelled/returned rate; high stockout fl...
231,PR0232,Electronics,76.953298,High,high cancelled/returned rate; high supplier delay
23,PR0024,Electronics,76.872528,High,high average discount; high supplier delay
39,PR0040,Fashion,76.129477,High,high cancelled/returned rate
8,PR0009,Electronics,75.417797,High,high stockout flag rate; high supplier delay
79,PR0080,Home,74.330827,High,high stockout flag rate; high supplier delay; ...
100,PR0101,Home,71.837877,High,unusual combination of business metrics


In [66]:
def suggest_action(row):
    actions = []

    if row["negative_margin_rate"] > 0.50:
        actions.append("review pricing and discount rules")

    if row["failure_rate"] > 0.20:
        actions.append("investigate cancellations and returns")

    if row["stockout_rate"] > 0.25:
        actions.append("review inventory policy")

    if row["avg_supplier_delay"] > product_risk["avg_supplier_delay"].quantile(0.75):
        actions.append("review supplier reliability")

    if not actions:
        actions.append("investigate unusual product behaviour")

    return "; ".join(actions)

In [67]:
product_risk["recommended_action"] = product_risk.apply(
    suggest_action,
    axis=1
)

In [68]:
product_risk_export = (
    product_risk
    .sort_values(
        "risk_score",
        ascending=False
    )
)

product_risk_export.to_csv(
    "product_risk.csv",
    index=False
)

print("Saved product_risk.csv")

Saved product_risk.csv


In [69]:
failure_by_category.to_csv(
    "failure_by_category.csv",
    index=False
)

failed_value_by_category.to_csv(
    "failed_value_by_category.csv",
    index=False
)

margin_by_category.to_csv(
    "margin_by_category.csv",
    index=False
)

margin_by_discount.to_csv(
    "margin_by_discount.csv",
    index=False
)

In [70]:
campaigns.loc[
    campaigns["campaign_invalid"]
].to_csv(
    "campaign_quality_issues.csv",
    index=False
)

In [71]:
inventory.loc[
    inventory["stockout_above_threshold"]
    | inventory["zero_stock_not_flagged"]
].to_csv(
    "inventory_quality_issues.csv",
    index=False
)

In [72]:
overview_metrics = pd.DataFrame({
    "metric": [
        "total_transaction_value",
        "cancelled_value",
        "returned_value",
        "combined_exposure",
        "exposure_pct",
        "negative_margin_rate",
        "invalid_campaign_rows",
        "stockout_above_threshold",
        "zero_stock_not_flagged"
    ],
    "value": [
        tx["revenue"].sum(),
        cancelled_value,
        returned_value,
        combined_exposure,
        exposure_pct,
        negative_margin_rate,
        campaigns["campaign_invalid"].sum(),
        inventory["stockout_above_threshold"].sum(),
        inventory["zero_stock_not_flagged"].sum()
    ]
})

overview_metrics.to_csv(
    "overview_metrics.csv",
    index=False
)

display(overview_metrics)

,metric,value
0,total_transaction_value,9.390803e+06
1,cancelled_value,8.618496e+05
2,returned_value,7.603223e+05
3,combined_exposure,1.622172e+06
4,exposure_pct,1.817790e+01
5,negative_margin_rate,3.793889e+01
6,invalid_campaign_rows,1.710000e+02
7,stockout_above_threshold,6.480000e+02
8,zero_stock_not_flagged,1.000000e+01
